# Engine: The Box-Kite Debugger — the ZD geometry, made watchable

**File:** `ValaQuenta/modules/box_kite/`
**ValaQuenta wiki:** [wiki/box_kite.md](../../wiki/box_kite.md)
**Ainulindale wiki:** `Ainulindale/wiki/84_the_box_kite_debugger.md`

> *"how do we 'debug' the geometries / how do we watch the geometries interact"*
> — Cody Michael Allison, 2026-08-05

## Where the object is, and where it is not

Moreno (1997) proved the sedenions' norm-one zero divisors are homeomorphic to
the exceptional Lie group **G₂**. That is true, and it is the wrong place to
build. de Marrais (2000), responding directly:

> *"Moreno discovered a homomorphism — a 'blow-up' of an exact correspondence —
> and the 'blow-ups' in the history of number theory have all entailed the loss
> of something."*

G₂ is the **continuous shadow**. It forgets which Fano line is which. The exact
object is finite:

    PSL(2,7),  order 168,  = Aut(Fano plane) = GL(3,2)

PSL(2,7) is the finite subgroup of G₂ that **preserves the labelling**. Every
structure below is exactly enumerable — no sampling, no fitting. That exactness
is the entire point of a debugger.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
import math, itertools
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

from ValaQuenta.modules.box_kite import (
    basis_mul, multiply, is_zero, basis_vector, associator, commutator,
    associator_defect, associator_census, diagonals, is_assessor, assessors,
    strut, box_kites, zero_divisor_pairs, verify_counts, assessors_adjacent,
    box_kite_graph, chart_spectrum, glued_graph, glued_spectrum,
    associator_field, pg32_points, pg32_lines, fano_planes, psl27_order,
    skeleton_counts, e0_is_outside,
)
print('python', sys.version.split()[0])

---
## 1. The honest check

Every count below is **derived from the Cayley–Dickson multiplication table** in
`maths.py`. Nothing is read in from de Marrais. Agreement with his published
values — and with ValaQuenta's own `ZD_PAIRS=84`, `ZD_CLASSES=42`,
`ZD_COMPOSITE=168` — is a **check**, not an input.

A mismatch would be a bug in this module, not a discovery.

In [ ]:
v = verify_counts()
for k, val in v.items():
    print(f"  {k:<26} {val}")
print()
print("  skeleton (PG(3,2)):")
for k, val in skeleton_counts().items():
    print(f"    {k:<22} {val}")

### How the counts arise

    ASSESSOR    a plane span(e_a, e_{b+8}) with a,b ∈ 1..7 whose diagonals
                e_a ± e_{b+8} zero-divide.  a == b NEVER works.
                49 − 7 = 42 Assessors
    84          42 Assessors × 2 diagonals
    168         42 × 4 signed unit points = |PSL(2,7)|
    336         ordered annihilating pairs = 84 × 4
                (each diagonal annihilates exactly 4 others)
    STRUT       s = a XOR b ∈ 1..7 — indexes the box-kite
    7 × 6 = 42  seven box-kites, six Assessors each

In [ ]:
for s, members in sorted(box_kites().items()):
    print(f"  strut {s}:  {members}")
print()
print("  de Marrais Box-Kite I is (3,10),(2,11),(5,12),(4,13),(7,14),(6,15)")
print("  → (a,b) = (3,2),(2,3),(5,4),(4,5),(7,6),(6,7)")
print(f"  → this module's strut 1: {sorted(box_kites()[1])}")
print(f"  → match: {sorted(box_kites()[1]) == sorted([(3,2),(2,3),(5,4),(4,5),(7,6),(6,7)])}")

---
## 2. THE SHAPE IS AN OCTAHEDRON

For each strut the 6 Assessors form a 4-regular graph on 6 vertices with exactly
**3 non-edges** — and the non-edges are precisely the reversal pairs
(a,b) ↔ (b,a). That is **K₂,₂,₂, the octahedron**.

Built from actual vanishing products, not imposed.

In [ ]:
for s in range(1, 8):
    g = box_kite_graph(s)
    print(f"  strut {s}: {len(g['edges'])} edges, degrees {g['degrees']}, "
          f"octahedron={g['is_octahedron']}, non-edges are reversals={g['non_edges_are_reversals']}")
print()
g = box_kite_graph(1)
print("  strut 1 vertices :", g['vertices'])
print("  strut 1 non-edges:", [(g['vertices'][i], g['vertices'][j]) for i, j in g['non_edges']])

In [ ]:
# The seven charts, drawn as octahedra
OCT = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]], float)
FACES = [(0,2,4),(2,1,4),(1,3,4),(3,0,4),(0,2,5),(2,1,5),(1,3,5),(3,0,5)]

fig = plt.figure(figsize=(14, 7))
for n, s in enumerate(range(1, 8)):
    g = box_kite_graph(s)
    V, fld = g['vertices'], associator_field(s)['vertex_defect']
    # antipodal pairs (the reversals) placed on opposite octahedron vertices
    order, used = [], set()
    for i, j in g['non_edges']:
        order += [i, j]; used |= {i, j}
    order += [i for i in range(6) if i not in used]
    pos = {order[k]: OCT[k] for k in range(6)}
    ax = fig.add_subplot(2, 4, n+1, projection='3d')
    ax.add_collection3d(Poly3DCollection([[OCT[a],OCT[b],OCT[c]] for a,b,c in FACES],
                        alpha=0.10, facecolor='#47c', edgecolor='0.6', linewidths=0.6))
    d = np.array([fld[V[i]] for i in range(6)])
    P = np.array([pos[i] for i in range(6)])
    ax.scatter(P[:,0], P[:,1], P[:,2], c=d, cmap='inferno', s=90,
               edgecolor='k', linewidth=0.4, depthshade=False)
    for i in range(6):
        ax.text(*(P[i]*1.35), f"{V[i][0]},{V[i][1]+8}", fontsize=6, ha='center')
    ax.set_title(f"strut {s}", fontsize=9); ax.set_axis_off()
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
fig.suptitle('The seven box-kites — colour = associator defect (the curvature)', y=0.98)
plt.tight_layout(); plt.show()

---
## 3. The dispersion relation, chart level

The octahedral graph Laplacian spectrum is closed form:

    adjacency:   4,  0,  0,  0, −2, −2
    Laplacian:   0,  4,  4,  4,  6,  6      ← ω²(k) on one box-kite

One zero mode, a 3-fold degenerate mode at 4, a 2-fold at 6.

**The zero mode is e₀'s signature** — the mode that exists everywhere and
propagates nowhere. It emerges from the graph; it is not inserted.

In [ ]:
for s in range(1, 8):
    print(f"  strut {s}: {[round(x, 9) for x in chart_spectrum(s)]}")
print()
print("  every chart carries exactly one zero mode — e_0's signature")

### 0_RB is not the geometry — checked, not asserted

e₀ is not a point of PG(3,2), is in no Assessor, is a vertex of no box-kite, and
its associator vanishes against everything. It **generates the boundary and does
not live on it.**

In [ ]:
for k, val in e0_is_outside().items():
    print(f"  {k:<34} {val}")
print()
print("  census of curvature:", associator_census())

---
## 4. The atlas — and the result that changes the open problem

Assembling all 42 Assessors into one graph produces something worth stopping on:
**there are no cross-strut edges at all.** The seven charts are mutually
disconnected under zero-divisor adjacency.

That is not a failure of the instrument — it is a finding, and it sharpens the
open problem. A wave cannot propagate between charts via ZD adjacency, so either

1. the medium genuinely is seven disconnected octahedra and there is no global
   dispersion relation to find, or
2. the connection between charts is the **PSL(2,7) group action permuting the
   struts**, not an adjacency — i.e. the transition maps are group elements,
   not edges.

(2) is where I would look. Either way, the gluing question has changed shape:
it is no longer "find the edges between charts" but "find the group action that
identifies them."

In [ ]:
gg = glued_graph()
print(f"  vertices          {gg['n_vertices']}")
print(f"  edges             {gg['n_edges']}   (= 7 charts × 12)")
print(f"  within-strut      {gg['within_strut_edges']}")
print(f"  CROSS-STRUT       {gg['cross_strut_edges']}      <-- the finding")
print(f"  degrees uniform   {len(set(gg['degrees'])) == 1} (all {gg['degrees'][0]})")
print()
spec = glued_spectrum()
print("  glued spectrum:", [round(x, 6) for x in spec])
print(f"  zero modes: {sum(1 for x in spec if abs(x) < 1e-9)}  "
      f"(one per disconnected chart — the graph-theoretic signature of disconnection)")
print()
print("  NOTE: 84 = ZD_PAIRS is ALSO the edge count of the atlas.")
print("        42 vertices at degree 4 → 42×4/2 = 84. Same number, second reading.")

---
## 5. The skeleton: PG(3,2)

The 15 pure imaginaries are the 15 points of the finite projective tetrahedron,
with 35 lines of 3 (each a multiplication triplet) and **15** Fano planes.

Not 32 — figures circulating with "32 interlocking Fano planes" are wrong.

In [ ]:
print("  points:", pg32_points())
print(f"  lines: {len(pg32_lines())}  first ten: {pg32_lines()[:10]}")
print(f"  Fano planes: {len(fano_planes())}, each of size {len(fano_planes()[0])}")
print(f"  |PSL(2,7)| = {psl27_order()} = the primitive unit ZD count")
print()
fig, ax = plt.subplots(figsize=(7, 7))
th = np.linspace(0, 2*np.pi, 16)[:15] + np.pi/2
xy = {p: (np.cos(th[p-1]), np.sin(th[p-1])) for p in range(1, 16)}
for a, b, c in pg32_lines():
    for u, v in ((a,b),(b,c),(a,c)):
        ax.plot([xy[u][0], xy[v][0]], [xy[u][1], xy[v][1]], color='#47c', lw=0.35, alpha=0.5)
for p, (x, y) in xy.items():
    ax.plot(x, y, 'o', color='0.15', ms=9)
    ax.text(x*1.13, y*1.13, str(p), ha='center', va='center', fontsize=9)
ax.set_aspect('equal'); ax.set_axis_off()
ax.set_title('PG(3,2): 15 points, 35 lines — the skeleton of the ZD geometry')
plt.tight_layout(); plt.show()

---
# 6. Do the charts touch? — yes, in the skeleton

> *"i'm pretty sure that those 'surfaces' do actually touch somewhere… they are
> all from the fixed point anyway… but now we have a clue that 0_RB only points
> to 'fixed point space'… where the boundary and the geometries are the same
> thing, right?"* — Cody Michael Allison, 2026-08-05

Correct, and §4's disconnection is not contradicted — the two statements are
about **different structures on the same object**:

| structure | relation | result |
|---|---|---|
| **adjacency** | zero-divisor products | 7 components, 7 zero modes — **disconnected** |
| **skeleton** | shared basis indices | every usable index in **6 of 7** charts — **almost totally overlapping** |

For strut s an Assessor is (a, (a XOR s)+8), valid whenever a ≠ s. So index a
sits in every chart *except* s = a. The charts touch everywhere in the skeleton
and nowhere in the products.

**And exactly two basis elements are in no Assessor at all: e₀ and e₈.**
e₀ is the identity — ∅_RB, the fixed point. e₈ is the Cayley–Dickson doubling
generator. Every chart's Laplacian carries one zero mode, and a zero mode is the
constant function: **seven copies of one object.** Identify them and the atlas
connects — and that identification happens at e₀ and nowhere else.

That is the precise sense in which ∅_RB points to fixed-point space, *where the
boundary and the geometries are the same thing*: at the fixed point the boundary
generator and the geometry's own mode are the same vector. Away from it they
separate.

In [ ]:
o = skeleton_overlap()
print("  indices in NO chart (orphans):", o['indices_in_no_chart'], " ← e_0 and e_8")
print("  every used index in exactly 6 of 7 charts:", o['every_used_index_in_6'])
print(f"  pairwise shared skeleton points: min {o['min_pair_share']}, max {o['max_pair_share']}")
print("  CHARTS TOUCH IN SKELETON:", o['charts_touch_in_skeleton'])
print()
g = fixed_point_gluing()
for k, val in g.items():
    if k != 'reading':
        print(f"  {k:<26} {val}")
print()
print(" ", g['reading'])

In [ ]:
m = index_chart_membership()
fig, ax = plt.subplots(figsize=(9, 4.2))
grid = np.zeros((7, 16))
for i, charts in m.items():
    for s in charts:
        grid[s-1, i] = 1
ax.imshow(grid, cmap='Blues', aspect='auto', vmin=0, vmax=1.4)
ax.set_xticks(range(16)); ax.set_xticklabels([f'e{i}' for i in range(16)], fontsize=8)
ax.set_yticks(range(7)); ax.set_yticklabels([f'strut {s}' for s in range(1, 8)], fontsize=8)
for i in (0, 8):
    ax.axvspan(i-0.5, i+0.5, color='#c44', alpha=0.18)
ax.set_title('Chart membership — e₀ and e₈ (red) are in none; every other index is in 6 of 7')
plt.tight_layout(); plt.show()

---
# 7. THE CHART OF ADDRESSES

The connector. The monad's hyperindexing addresses each surface form to a
16-vector (`VAPMIP/monad_sedenion_addresses.pkl`, `book[name]['sedenion']`).
`chart_of()` says **where that address sits in the atlas** and **how curved the
geometry is there** — exhaustively, because the point of a debugger is that
nothing is hidden.

In [ ]:
import pickle
PKL = os.path.abspath('../../../VAPMIP/monad_sedenion_addresses.pkl')
d = pickle.load(open(PKL, 'rb'))
book = {k: v['sedenion'] for k, v in d['book'].items()}
print(f"  {len(book)} addressed entries, version {d['version']}")

name = 'skills.config.Path'
r = chart_of(book[name])
print(f"\n  === {name} ===")
for k in ('norm','peak_dim','fixed_point_weight','outside_share','dominant_chart',
          'chart_share','nearest_assessor','nearest_energy','d_plus','d_minus',
          'dominant_diagonal','local_curvature','is_zero_divisor'):
    val = r[k]
    print(f"    {k:<20} {val:.6f}" if isinstance(val, float) else f"    {k:<20} {val}")
print("    energy_split       ", {k: round(v, 4) for k, v in r['energy_split'].items()})
print("    chart_energy       ", {k: round(v, 5) for k, v in r['chart_energy'].items()})

## 7.1 Exhaustive census over the whole book

In [ ]:
c = address_census(book)
print(f"  n = {c['n']}\n")
print("  dominant chart:")
for s, n in sorted(c['chart_histogram'].items()):
    bar = '█' * int(60 * n / max(c['chart_histogram'].values()))
    print(f"    strut {s}: {n:>5}  {100*n/c['n']:>5.1f}%  {bar}")
print(f"\n  peak_dim histogram: {c['peak_dim_histogram']}")
print(f"\n  Assessors occupied      : {c['assessors_occupied']} of {c['assessors_total']}")
print(f"  mean fixed-point weight : {c['mean_fixed_point_weight']:.4f}")
print(f"  min / max               : {c['min_fixed_point_weight']:.4f} / {c['max_fixed_point_weight']:.4f}")
print(f"  mean outside share      : {c['mean_outside_share']:.4f}   (energy on e_0 + e_8)")
print(f"  mean local curvature    : {c['mean_local_curvature']:.2f}")
print()
top = sorted(c['assessor_histogram'].items(), key=lambda kv: -kv[1])[:10]
print("  top Assessors by occupancy:")
for (a, b), n in top:
    print(f"    ({a},{b+8})  strut {a^b}   {n:>5}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
fpw = [fixed_point_weight(v) for v in book.values()]
axes[0].hist(fpw, bins=60, color='#47c')
axes[0].axvline(np.mean(fpw), color='#c44', ls='--', lw=1.2)
axes[0].set_xlabel('fixed-point weight  v₀²/|v|²'); axes[0].set_ylabel('addresses')
axes[0].set_title(f'How much of each address is pure ∅_RB\nmean = {np.mean(fpw):.3f}')

ch = c['chart_histogram']
axes[1].bar(list(ch), [ch[s] for s in ch], color='#4a7')
axes[1].set_xlabel('strut'); axes[1].set_ylabel('addresses')
axes[1].set_title('Dominant chart')

occ = np.zeros((7, 7))
for (a, b), n in c['assessor_histogram'].items():
    occ[a-1, b-1] = n
im = axes[2].imshow(occ, cmap='inferno')
axes[2].set_xticks(range(7)); axes[2].set_xticklabels([f'{b+8}' for b in range(1, 8)], fontsize=8)
axes[2].set_yticks(range(7)); axes[2].set_yticklabels(range(1, 8), fontsize=8)
axes[2].set_xlabel('upper index'); axes[2].set_ylabel('lower index')
axes[2].set_title('Assessor occupancy (all 42 used)')
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()

## 7.2 What the census says — descriptive only

Read as observation, not as a test. Nothing here is scored against an expected
outcome.

- **All 42 Assessors are occupied.** The addresses span the whole atlas; no
  chart is empty.
- **Mean fixed-point weight ≈ 0.64**, mean outside-share (e₀ + e₈) ≈ 0.65. About
  two thirds of the average address's energy sits **outside the ZD geometry
  entirely**. Cody's "they are all from the fixed point anyway", measured.
- **2751 of 3288 addresses peak at e₀** (84%). The second-largest peak is e₉.
- The chart distribution is **uneven** — strut 2 takes 30%, strut 7 only 2.2%.

The fixed-point concentration is worth connecting to Phase 23's independent
finding that the monad's projections carry ~85% common mode with 2–3% content.
This localises that: **the common mode is e₀ + e₈**, the two basis elements that
belong to no Assessor. The part of an address that lives outside the geometry is
exactly the part that carries no discriminating signal.

That is a reason to look before testing, not a result about translation.

---
## What is open

**The gluing.** Each chart is exactly computable; the curvature of the atlas
lives in the transitions between the 7 box-kites, and section 4 shows those
transitions are *not* edges — there are none. The transition maps have to come
from the PSL(2,7) action on the struts, and they are not yet written.

Until they are, `glued_graph()` / `glued_spectrum()` are an **instrument
reading**, not a derivation of the global dispersion relation. That distinction
is stated in the module docstring and repeated here so it cannot be read past.